# 04 - Player Analysis (Batters)

In [ ]:
import pandas as pd
import numpy as np

matches = pd.read_csv('data/matches_clean.csv')
deliveries = pd.read_csv('data/deliveries_clean.csv')
print(matches.shape, deliveries.shape)

## Top run scorers

In [ ]:
runs = deliveries.groupby('batter')['batsman_runs'].sum().sort_values(ascending=False)
runs.head(10)

## Balls faced, strike rate (excluding wides)

In [ ]:
legal = deliveries[deliveries['extras_type'] != 'wides']
balls = legal.groupby('batter').size()

bat = pd.DataFrame({'runs': runs, 'balls': balls})
bat['strike_rate'] = (bat['runs'] / bat['balls'] * 100).round(2)
bat.sort_values('runs', ascending=False).head(10)

## Best strike rate (min 500 balls)

In [ ]:
bat[bat['balls'] >= 500].sort_values('strike_rate', ascending=False).head(10)

## Fours and sixes

In [ ]:
fours = deliveries[deliveries['batsman_runs'] == 4].groupby('batter').size()
sixes = deliveries[deliveries['batsman_runs'] == 6].groupby('batter').size()
print('Most fours'); print(fours.sort_values(ascending=False).head(5))
print('\nMost sixes'); print(sixes.sort_values(ascending=False).head(5))

## Highest individual scores in an innings

In [ ]:
ind = deliveries.groupby(['match_id', 'batter'])['batsman_runs'].sum().reset_index()
ind.sort_values('batsman_runs', ascending=False).head(10)

In [ ]:
print('Centuries:'); print(ind[ind['batsman_runs'] >= 100]['batter'].value_counts().head(10))
print('\nFifties  :'); print(ind[(ind['batsman_runs'] >= 50) & (ind['batsman_runs'] < 100)]['batter'].value_counts().head(10))

## Batting average (runs / dismissals)

In [ ]:
outs = deliveries[deliveries['player_dismissed'] != 'None'].groupby('player_dismissed').size()
bat['dismissals'] = outs
bat['average'] = (bat['runs'] / bat['dismissals']).round(2)
bat[bat['runs'] >= 1500].sort_values('average', ascending=False).head(10)

## Player of the match awards

In [ ]:
matches[matches['player_of_match'] != 'Not Awarded']['player_of_match'].value_counts().head(10)

## Orange cap (top scorer per season)

In [ ]:
d = deliveries.merge(matches[['id', 'season']], left_on='match_id', right_on='id')
season_runs = d.groupby(['season', 'batter'])['batsman_runs'].sum().reset_index()
season_runs.loc[season_runs.groupby('season')['batsman_runs'].idxmax()]